For NUS:
    keep:
        "department"
        "description"
        "faculty"
        "moduleCode"
        "title"
        
    ignore:
        "gradingBasisDescription"
        "moduleCredit"
        "semesterData"
        "workload"


For NTU:
    keep:
        "code" 
        "title" 
        "description" 
        "departmentMaintaining"

    ignore:
        "academicUnits"
        "examSchedule" 
        "gradeType" 
        "prerequisites" 
        "notAvailableToProgramme" (eg not available to EEE)
        "url" 
        "details" 


For SUTD:
    keep:
        "code"
        "title" 
        "course_type" (core, electives, etc)
        "description" 
        *missing department

    ignore:
        "terms" 
        "credits" 
        "url" 
        "description_found" (T/F)


extract skills from the descriptions

courses:
|university| department| faculty| code| title| course_type| description| skills|

jobs:
    keep: 
        "title"
        "description"
        "minimumYearsExperience"
        "shiftPattern"
        "skills" (keep everything in the skills section)
            "uuid": "11ff9f68afb6b8b5b8eda218d7c83a65",
            "confidence": null,
            "isKeySkill": null
        "otherRequirements"
        "ssocCode" (job industry/sector)
        "occupationId"
        "ssocVersion"
        "workingHours"
        "numberOfVacancies"
        "categories"
            "id"
            "category"
        "employmentTypes" 
            "id" 
            "employmentType" 
        "positionLevels"
            "id": 11,
            "position" 
        "ssecEqa" 
            (Singapore Standard Educational Classification (SSEC) Educational Qualification Assessment code, indicating the education level. 
            eg "69" typically corresponds to a Bachelor's Degree with Honours.)
        "ssecFos" 
            (Field of Study code, 
            eg "0212" corresponds to "Music and Performing Arts")
        "ssicCode" 
            (It classifies the type of industry/business sector the company operates in.)
            (SSOC = the job classification (Laboratory Technician, Finance Manager, etc.)
            (SSIC = the company/industry classification (Manufacturing, Education, etc.))
        "ssicCode2020" 
         "salary": {
                    "maximum": 3500,
                    "minimum": 2500,
                    "type": {
                    "id": 4,
                    "salaryType": "Monthly"
                    }
        "jobTitles"

    keep but tbc: (if all NA/nulls -> drop)
        "schemes" 
        "flexibleWorkArrangements"

    ignore:
        "uuid" (post id? user id?)
        "sourceCode" 
        "psdUrl"
        "status"
        "postedCompany" 
            "uen" 
                ("uen" stands for Unique Entity Number, a unique identifier assigned to all business entities (companies, organizations, partnerships) registered in Singapore. In the example, "52875677W" is the UEN for SYMPHONY MUSIC SCHOOL. It's used to uniquely identify and track companies in Singapore's business registry.)
            "description" (company description)
            "name" (company name)
        "employeeCount" 
        "companyUrl"
        "lastSyncDate": "2026-01-30T02:57:44.000Z",
        "_links"
        "logoFileName" 
        "logoUploadPath" 
        "responsiveEmployer": {
      "isResponsive": false
    }



# NUS

## Load Data

In [2]:
# import necessary libraries
import json5
import pandas as pd
from pathlib import Path

In [3]:
# load output NUSModsInfo.json file
nus_file_path = Path("../../data/NUSModsInfo.json")

with open(nus_file_path, "r", encoding="utf-8") as f:
    nus_data = json5.load(f)

In [4]:
# Extract relevant fields
nus_fields = ["department", "description", "faculty", "moduleCode", "title"]

nus_filtered_data = [
    {key: item.get(key) for key in nus_fields}
    for item in nus_data
]

In [5]:
# Create DataFrame
nus_df = pd.DataFrame(nus_filtered_data)

# Preview
nus_df.head()

,department,description,faculty,moduleCode,title
0,NUS Medicine Dean's Office,Leadership is fundamental to the success of in...,Yong Loo Lin Sch of Medicine,ABM5001,Leadership in Biomedicine
1,NUS Medicine Dean's Office,This course serves as a concept-based introduc...,Yong Loo Lin Sch of Medicine,ABM5002,Advanced Biostatistics for Research
2,NUS Medicine Dean's Office,This course will furnish students with a thoro...,Yong Loo Lin Sch of Medicine,ABM5003,Biomedical Innovation & Enterprise
3,NUS Medicine Dean's Office,This course encompasses research projects rele...,Yong Loo Lin Sch of Medicine,ABM5004,Capstone Project
4,NUS Medicine Dean's Office,Advanced immunological applications play impor...,Yong Loo Lin Sch of Medicine,ABM5101,Applied Immunology


## Data Cleaning

### Data Exploration

In [6]:
# check for missing values
print(nus_df.isnull().sum())

department     103
description      0
faculty          0
moduleCode       0
title            0
dtype: int64


In [7]:
nus_df.info()

<class 'pandas.DataFrame'>
RangeIndex: 20517 entries, 0 to 20516
Data columns (total 5 columns):
 #   Column       Non-Null Count  Dtype
---  ------       --------------  -----
 0   department   20414 non-null  str  
 1   description  20517 non-null  str  
 2   faculty      20517 non-null  str  
 3   moduleCode   20517 non-null  str  
 4   title        20517 non-null  str  
dtypes: str(5)
memory usage: 801.6 KB


In [8]:
# unique values in department and faculty
print("Unique departments:", nus_df["department"].nunique())
print("Unique faculties:", nus_df["faculty"].nunique()) 

Unique departments: 109
Unique faculties: 25


In [9]:
# get unique faculties 
unique_faculties = nus_df["faculty"].unique()
print("Unique faculties:", unique_faculties)

Unique faculties: <StringArray>
[     'Yong Loo Lin Sch of Medicine', 'College of Design and Engineering',
               'NUS Business School',           'Arts and Social Science',
                               'NUS',                         'Computing',
                               'Law',       'Cont and Lifelong Education',
                           'Science',     'Non-Faculty-based Departments',
                         'Dentistry',         'YST Conservatory of Music',
      'Multi Disciplinary Programme',                           'NUS-ISS',
               'Residential College',    'Temasek Defence Sys. Institute',
         'Risk Management Institute',                       'NUS College',
       'SSH School of Public Health',           'Duke-NUS Medical School',
               'NUS Graduate School',           'Logistics Inst-Asia Pac',
    'Mechanobiology Institute (MBI)',       'LKY School of Public Policy',
                  'Yale-NUS College']
Length: 25, dtype: str


In [10]:
# faculty level data distribution
faculty_counts = nus_df["faculty"].value_counts()
print(faculty_counts)

faculty
Arts and Social Science              5679
Law                                  2878
College of Design and Engineering    2823
Science                              1695
NUS Business School                  1642
Yale-NUS College                     1485
Computing                             541
Yong Loo Lin Sch of Medicine          517
Cont and Lifelong Education           425
Duke-NUS Medical School               416
LKY School of Public Policy           377
YST Conservatory of Music             373
Non-Faculty-based Departments         366
NUS College                           337
Residential College                   261
SSH School of Public Health           208
NUS                                   154
NUS-ISS                                82
NUS Graduate School                    65
Dentistry                              59
Temasek Defence Sys. Institute         51
Risk Management Institute              48
Multi Disciplinary Programme           15
Mechanobiology Institute (

### Filter for undergraduate courses only

In [11]:
# Keep only undergraduate level courses: codes where the first digit after letters is 0-4
nus_df = nus_df[nus_df['moduleCode'].str.match(r'^[A-Z]+[0-4]')]

nus_df

,department,description,faculty,moduleCode,title
27,Accounting,The course provides an introduction to account...,NUS Business School,ACC1701,Accounting for Decision Makers
28,Accounting,The course provides an introduction to account...,NUS Business School,ACC1701A,Accounting for Decision Makers
29,Accounting,The course provides an introduction to account...,NUS Business School,ACC1701B,Accounting for Decision Makers
30,Accounting,The course provides an introduction to account...,NUS Business School,ACC1701C,Accounting for Decision Makers
31,Accounting,The course provides an introduction to account...,NUS Business School,ACC1701D,Accounting for Decision Makers
...,...,...,...,...,...
20512,Biological Sciences,In addition to having an academic science foun...,Science,ZB3312,Enhanced Undergraduate Professional Internship...
20513,Biological Sciences,In addition to having an academic science foun...,Science,ZB3313,Undergraduate Professional Internship Programm...
20514,Biological Sciences,This is a seminar-style course based on the li...,Science,ZB4171,Advanced Topics in Bioinformatics
20515,Biological Sciences,Not Available,Science,ZB4199,Honours Project in Computational Biology


In [12]:
# faculty level data distribution
faculty_counts = nus_df["faculty"].value_counts()
print(faculty_counts)

faculty
Arts and Social Science              4631
Yale-NUS College                     1485
College of Design and Engineering    1374
Science                              1172
NUS Business School                   796
Law                                   783
Cont and Lifelong Education           403
Computing                             350
YST Conservatory of Music             337
NUS College                           337
Non-Faculty-based Departments         333
Residential College                   261
Yong Loo Lin Sch of Medicine          174
NUS                                   122
SSH School of Public Health            61
Dentistry                              50
NUS-ISS                                19
Multi Disciplinary Programme           15
Duke-NUS Medical School                 3
Temasek Defence Sys. Institute          2
Name: count, dtype: int64


In [13]:
# filter out post graduate level courses: faculty is for post graduate level courses, which are not relevant to our analysis
target_faculties = [
    "Temasek Defence Sys. Institute",
     "Cont and Lifelong Education",
     "Duke-NUS Medical School"
]

filtered_nus_df = nus_df[~nus_df["faculty"].isin(target_faculties)]

### Handling NA Values

In [14]:
# count data with null description
null_description_count = filtered_nus_df["description"].isnull().sum()
print("Number of courses with null description:", null_description_count)

Number of courses with null description: 0


In [15]:
# Distribution of description data
description_counts = filtered_nus_df["description"].value_counts()
print(description_counts)

description
Not Available                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                               3664
UTOP aims to train undergraduates to acquire and promulgate “teaching” skills. When pursuing such traineeship in teaching under UTOP, students can develop their teaching skills in a more systematic manner, and become be

some short descriptions such as: NIL, Exchange Course - YUS (1 unit), Not Applicable 
are not meaningful for analysis. we would remove those data

In [16]:
# Keep only rows where description has 10 or more words
filtered_nus_df = filtered_nus_df[filtered_nus_df['description'].str.split().str.len() >= 10]

In [17]:
filtered_nus_df.info()

<class 'pandas.DataFrame'>
Index: 8499 entries, 27 to 20516
Data columns (total 5 columns):
 #   Column       Non-Null Count  Dtype
---  ------       --------------  -----
 0   department   8499 non-null   str  
 1   description  8499 non-null   str  
 2   faculty      8499 non-null   str  
 3   moduleCode   8499 non-null   str  
 4   title        8499 non-null   str  
dtypes: str(5)
memory usage: 398.4 KB


there is no more NA data in the dataset

## Data Transformation

In [18]:
filtered_nus_df["University"] = "NUS"

In [19]:
filtered_nus_df.head()

,department,description,faculty,moduleCode,title,University
27,Accounting,The course provides an introduction to account...,NUS Business School,ACC1701,Accounting for Decision Makers,NUS
28,Accounting,The course provides an introduction to account...,NUS Business School,ACC1701A,Accounting for Decision Makers,NUS
29,Accounting,The course provides an introduction to account...,NUS Business School,ACC1701B,Accounting for Decision Makers,NUS
30,Accounting,The course provides an introduction to account...,NUS Business School,ACC1701C,Accounting for Decision Makers,NUS
31,Accounting,The course provides an introduction to account...,NUS Business School,ACC1701D,Accounting for Decision Makers,NUS


# NTU

## Load Data

In [47]:
# load output ntuCourseInfo.json file
ntu_file_path = Path("../../data/ntuCourseInfo.json")

with open(ntu_file_path, "r", encoding="utf-8") as f:
    ntu_data = json5.load(f)

In [48]:
# Extract relevant fields
ntu_fields =  ["code", "title", "description", "departmentMaintaining"]

ntu_filtered_data = [
    {key: item.get(key) for key in ntu_fields}
    for item in ntu_data
]

In [49]:
# Create DataFrame
ntu_df = pd.DataFrame(ntu_filtered_data)

# Preview
ntu_df.head()

,code,title,description,departmentMaintaining
0,AAA08B,AAA08B FASHION & DESIGN: WEARABLE ART AS A SEC...,Develop process skills in expression and visua...,NIE
1,AAA08C,AAA08C EXPRESSIVE DRAWING: DEVELOPING PERSONAL...,Students will learn a brief history of express...,NIE
2,AAA08D,AAA08D ABSTRACT PAINTING: WHY IT'S HERE & HOW ...,Students will learn a brief history of abstrac...,NIE
3,AAA18D,AAA18D LIFE DRAWING,This course introduces drawing of the nude hum...,NIE
4,AAA18E,AAA18E DRAWING,This course introduces drawing at a basic leve...,NIE


## Data Cleaning

### Data Exploration

In [50]:
# check for missing values
print(ntu_df.isnull().sum())

code                      0
title                     0
description               0
departmentMaintaining    14
dtype: int64


In [51]:
ntu_df.info()

<class 'pandas.DataFrame'>
RangeIndex: 2117 entries, 0 to 2116
Data columns (total 4 columns):
 #   Column                 Non-Null Count  Dtype
---  ------                 --------------  -----
 0   code                   2117 non-null   str  
 1   title                  2117 non-null   str  
 2   description            2117 non-null   str  
 3   departmentMaintaining  2103 non-null   str  
dtypes: str(4)
memory usage: 66.3 KB


In [52]:
# unique values in department
print("Unique departments:", ntu_df["departmentMaintaining"].nunique())

Unique departments: 47


In [53]:
# faculty level data distribution
ntu_dept_counts = ntu_df["departmentMaintaining"].value_counts()
print(ntu_dept_counts)

departmentMaintaining
BUS          144
ADM          144
CSC(CE)      129
NIE          119
SOH          111
CS            96
EEE           96
ELH(SOH)      74
MATH(SPS)     71
CHIN(SOH)     67
ME            64
BS            62
LMS(SOH)      57
PSY(SSS)      56
EESS(ASE)     54
HIST(SOH)     53
MAT           53
ECON(SSS)     51
CBE           49
PPGA(SSS)     49
PHY(SPS)      49
PHIL(SOH)     48
SOC(SSS)      41
CEE           39
CHEM(CBE)     37
ACC           36
CE            36
BIE(CBE)      28
REP           25
SSM(NIE)      22
MS(CEE)       21
ENE(CEE)      20
AERO(ME)      20
CMED(BS)      15
NTC           14
BMS(BS)       13
SSS            9
ICC            8
SPS            6
BSB(BS)        4
IEM(EEE)       4
ROBO(ME)       3
BACF(CE)       2
BMS            1
BSB            1
CAO            1
ACDA(CE)       1
Name: count, dtype: int64


| Code          | School / Unit                                             |
| ------------- | --------------------------------------------------------- |
| **BUS**       | Nanyang Business School (NBS)                             |
| **ACC**       | Nanyang Business School (Accountancy)                     |
| **ADM**       | School of Art, Design and Media                           |
| **CS**        | School of Computer Science and Engineering (SCSE)         |
| **CSC(CE)**   | Computer Science (Computer Engineering track, SCSE)       |
| **CE**        | School of Civil and Environmental Engineering (CEE)       |
| **CEE**       | School of Civil and Environmental Engineering             |
| **ENE(CEE)**  | Environmental Engineering (CEE)                           |
| **MS(CEE)**   | Maritime Studies (CEE)                                    |
| **EEE**       | School of Electrical and Electronic Engineering           |
| **IEM(EEE)**  | Information Engineering & Media (EEE)                     |
| **ME**        | School of Mechanical and Aerospace Engineering            |
| **AERO(ME)**  | Aerospace Engineering (MAE)                               |
| **ROBO(ME)**  | Robotics (MAE)                                            |
| **CBE**       | School of Chemistry, Chemical Engineering & Biotechnology |
| **CHEM(CBE)** | Chemistry (CBE)                                           |
| **BIE(CBE)**  | Bioengineering (CBE)                                      |
| **MATH(SPS)** | School of Physical & Mathematical Sciences                |
| **PHY(SPS)**  | Physics (SPMS)                                            |
| **SPS**       | School of Physical & Mathematical Sciences                |
| **BS**        | School of Biological Sciences                             |
| **BMS(BS)**   | Biomedical Sciences (Biological Sciences)                 |
| **CMED(BS)**  | Chinese Medicine (Biological Sciences)                    |
| **BSB(BS)**   | Biological Sciences Programme                             |
| **BMS**       | Biological Sciences (general)                             |
| **BSB**       | Biological Sciences (general)                             |
| **SOH**       | School of Humanities                                      |
| **ELH(SOH)**  | English (Humanities)                                      |
| **HIST(SOH)** | History (Humanities)                                      |
| **PHIL(SOH)** | Philosophy (Humanities)                                   |
| **CHIN(SOH)** | Chinese (Humanities)                                      |
| **LMS(SOH)**  | Linguistics & Multilingual Studies                        |
| **SSS**       | School of Social Sciences                                 |
| **PSY(SSS)**  | Psychology                                                |
| **ECON(SSS)** | Economics                                                 |
| **SOC(SSS)**  | Sociology                                                 |
| **PPGA(SSS)** | Public Policy & Global Affairs                            |
| **EESS(ASE)** | Asian School of the Environment                           |
| **NIE**       | National Institute of Education                           |
| **SSM(NIE)**  | Sports Science & Management (NIE)                         |
| **REP**       | Renaissance Engineering Programme                         |
| **ICC**       | Interdisciplinary Collaborative Core                      |
| **NTC**       | NTU-wide / Core curriculum                                |
| **CAO**       | Likely programme-specific / administrative unit           |
| **ACDA(CE)**  | Programme under Civil Engineering                         |


### Filter for undergraduate courses only

In [54]:
# Keep only undergraduate level courses: codes where the first digit after letters is 0-4
ntu_df = ntu_df[ntu_df['code'].str.match(r'^[A-Z]+[0-4]')]

### Add department name

In [55]:
ntu_dept_file_path = Path("../../data/ntu_dept_mapping.xlsx")

ntu_dept_df = pd.read_excel(ntu_dept_file_path)

ntu_dept_df.head()

,short,department
0,ACC,Nanyang Business School (Accountancy)
1,ACDA(CE),Programme under Civil Engineering
2,ADM,"School of Art, Design and Media ..."
3,AERO(ME),Aerospace Engineering (MAE)
4,BACF(CE),Bachelor of Applied Computing in Finance (CE)


In [57]:
# add department mapping to ntu_df
ntu_df = ntu_df.merge(
    ntu_dept_df[["short", "department"]],
    left_on="departmentMaintaining",
    right_on="short",
    how="left"
).drop(columns=["short"])

In [59]:
ntu_df.rename(columns={"departmentMaintaining": "dept_code"}, inplace=True)

### Handling NA Values

In [65]:
# count data with null description
ntu_null_description_count = ntu_df["description"].isna().sum()
print("Number of courses with null description:", ntu_null_description_count)

Number of courses with null description: 0


In [62]:
# Distribution of description data
ntu_description_counts = ntu_df["description"].value_counts()
print(ntu_description_counts)

description
                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                            

In [68]:
# Keep only rows where description has 10 or more words
filtered_ntu_df = ntu_df[ntu_df['description'].str.split().str.len() >= 10]

In [69]:
filtered_ntu_df.info()

<class 'pandas.DataFrame'>
Index: 1818 entries, 0 to 1851
Data columns (total 5 columns):
 #   Column       Non-Null Count  Dtype
---  ------       --------------  -----
 0   code         1818 non-null   str  
 1   title        1818 non-null   str  
 2   description  1818 non-null   str  
 3   dept_code    1818 non-null   str  
 4   department   1818 non-null   str  
dtypes: str(5)
memory usage: 85.2 KB


## Data Transformation

In [70]:
filtered_ntu_df["University"] = "NTU"

In [72]:
filtered_ntu_df.head()

,code,title,description,dept_code,department,University
0,AAA08B,AAA08B FASHION & DESIGN: WEARABLE ART AS A SEC...,Develop process skills in expression and visua...,NIE,National Institute of Education,NTU
1,AAA08C,AAA08C EXPRESSIVE DRAWING: DEVELOPING PERSONAL...,Students will learn a brief history of express...,NIE,National Institute of Education,NTU
2,AAA08D,AAA08D ABSTRACT PAINTING: WHY IT'S HERE & HOW ...,Students will learn a brief history of abstrac...,NIE,National Institute of Education,NTU
3,AAA18D,AAA18D LIFE DRAWING,This course introduces drawing of the nude hum...,NIE,National Institute of Education,NTU
4,AAA18E,AAA18E DRAWING,This course introduces drawing at a basic leve...,NIE,National Institute of Education,NTU
